# Closed-form solutions of planar equivalent layer integral

### Things we have to import

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from gravmag import eqlayer, filters, transforms, convolve
from gravmag.models import rectangular_prism as rp
from gravmag import plot_functions as plf
from gravmag import constants as cts

### Model

In [2]:
# Create a model formed by a single prism
#model = np.array([[-500, 500, 0, 2000, 10, 1010]])
model = np.array([[-500, 500, 0, 2000, 300, 1010]])

density = np.array([2000]) # 2000 kg/m³ = 2 g/cm³

### Points were we compute the synthetic data

In [3]:
# number of points along x and y
shape = (100, 100)

# minimum x, maximum x, minimum y and maximum y
area = [-5000, 5000, -4000, 6000]

# coordinates 
xp = np.linspace(area[0], area[1], shape[0])
yp = np.linspace(area[2], area[3], shape[1])
yp, xp = np.meshgrid(yp, xp)

yp = np.ravel(yp)
xp = np.ravel(xp)

# constant vertical coordinate of the simulated gravity data
z_data = -150

zp = np.zeros_like(yp) + z_data

coordinates = np.vstack([xp, yp, zp])

In [4]:
# The grid of blocks and the order of each block in the BTTB
# are defined by data ordering (x- or y-oriented)

# The number of blocks is defined by the number of points along x-axis
N_blocks = shape[0]

# the number of points per block is defined by the number of points along y-axis
N_points_per_block = shape[1]

In [5]:
# Grid spacing
dx = (area[1] - area[0])/(shape[0]-1)
dy = (area[3] - area[2])/(shape[1]-1)

In [6]:
print(dx, dy)

101.01010101010101 101.01010101010101


### Synthetic data

In [7]:
data = rp.grav(coordinates=coordinates, prisms=model, density=density, field='z')

In [8]:
N_data = data.size

### Equivalent layer

In [9]:
z_t = np.min(model[:,4])
z_t

300

In [10]:
Dz = z_t - z_data
Dz

450

In [11]:
# layer z
z_layer = z_data + 0.9*Dz
z_layer

255.0

In [12]:
# kernel matrix associated with the equivalent layer
scale = cts.GRAVITATIONAL_CONST*cts.SI2MGAL
A = scale*eqlayer.kernel_matrix_monopoles(coordinates, z_layer, field='x')

#### Space-domain approach via linear system

In [13]:
#p_linsys = np.linalg.solve(A.T@A, A.T@data)
p_linsys = np.linalg.solve(A, data)

In [14]:
data_pre_linsys = A@p_linsys

In [15]:
misfit_linsys = np.sum((data - data_pre_linsys)**2)/N_data

In [16]:
misfit_linsys

6.368206059640727e-27

#### Fourier-domain approach via deconvolution

In [17]:
# symmetry of matrix A
symmetry_A = 'skew-symm'

# first column of the embeddin BCCB matrix
c0 = convolve.embedding_BCCB_first_column(
    b0=A[:,0], Q=N_blocks, P=N_points_per_block, symmetry=symmetry_A
)

# matrix containing the eigenvalues of matrix A
La = convolve.eigenvalues_BCCB(
    c0=c0, Q=N_blocks, P=N_points_per_block, ordering="row"
)

In [ ]:
p_deconvolution = convolve.product_BCCB_vector(
    L=1./La, Q=N_blocks, P=N_points_per_block, v=data, ordering="row"
)

In [ ]:
data_pre_deconvolution = A@p_deconvolution

In [ ]:
misfit_deconvolution = np.sum((data - data_pre_deconvolution)**2)/N_data

In [ ]:
misfit_deconvolution

### Gravitational field predicted by the equivalent layer

In [ ]:
# compute the predicted field and update the list
predicted_fields = [data_pre_deconvolution, data_pre_linsys]

### Plot the results

In [ ]:
true_fields = [data, data]

bounds, diffs = plf.bounds_diffs(predicted_fields, true_fields)

fields = plf.fields_list(predicted_fields, true_fields, diffs)

titles = ['predicted', 'true', 'difference',
          'predicted', 'true', 'difference']

windows = [i for i in range(1,7)]

plt.figure(figsize=(15,8))

for w, window in enumerate(windows):

    plt.subplot(2,3,window)
    plt.title(titles[w], fontsize=14)
    plt.axis('scaled')
    plt.contourf(
        yp.reshape(shape)*0.001, 
        xp.reshape(shape)*0.001, 
        fields[w].reshape(shape), 
        cmap='RdBu_r',
        vmin = -bounds[w], vmax = bounds[w]
    )
    plt.colorbar()
    plf.model_boundaries(model, m2km=True)
    plt.ylim(0.001*area[0], 0.001*area[1])
    plt.xlim(0.001*area[2], 0.001*area[3])
    plt.xlabel('y (km)', fontsize=12)
    plt.ylabel('x (km)', fontsize=12)

plt.tight_layout()

plt.show()